## READ ME FIRST

This notebook produces a weekly release from the `develop` branch of Cytoscape.

The weekly release consists of Install4j generated installers that can be put on a public server to be downloaded by testers. This script will NOT push any code changes or resources to GitHub or Nexus or update any additional resources like the manual or API docs. 

This notebook is configured for the build server on which it is run. Notes have been made where local environment variables have been set.

Before starting this process, it is a good idea to Restart and Clear Output for this notebook's Kernel.


### 1. Set up Build Environment

This sets up the build environment, including Java and Maven versions, and the root directory of the build.


In [1]:
from subprocess import Popen, PIPE
import os
import shutil

Set the notebook directory. Note that this is system dependent.

In [2]:
NOTEBOOK_DIR = '/home/cybuilder/cytoscape-admin-scripts'

print(NOTEBOOK_DIR)

/home/cybuilder/cytoscape-admin-scripts


Set the Java environment. This is Java 17 for Cytoscape 3.10.0 and above.

Note that this is system dependent, and may need to be changed to reflect the system's configuration.

In [3]:
%env JAVA_HOME=/opt/jdk-17

env: JAVA_HOME=/opt/jdk-17


Set the MAVEN_HOME environment variable, as well as point the path to the correct Maven binaries. 

Note that this is system dependent and may need to be changed to reflect the system's configuration.

In [4]:
%env MAVEN_HOME=/opt/maven
%env PATH=/opt/jdk-17/bin:/opt/apache-maven-3.6.0/bin/:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin

env: MAVEN_HOME=/opt/maven
env: PATH=/opt/jdk-17/bin:/opt/apache-maven-3.6.0/bin/:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin


Prepare the target directory and clone the cytoscape git repo.

In [5]:
print('Changing to directory: ' + NOTEBOOK_DIR)
os.chdir(NOTEBOOK_DIR)

# Point to build location (the directory to clone parent cytoscape into)
BUILD_PARENT_DIR = os.path.join(os.getcwd(), 'release-build')
if not os.path.exists(BUILD_PARENT_DIR):
    os.mkdir(BUILD_PARENT_DIR)
else:
    shutil.rmtree(BUILD_PARENT_DIR)
    os.mkdir(BUILD_PARENT_DIR)

os.chdir(BUILD_PARENT_DIR)
![[ -d cytoscape ]] || git clone https://github.com/cytoscape/cytoscape
CYTOSCAPE_ROOT_DIR = os.path.join(BUILD_PARENT_DIR, 'cytoscape')
CYTOSCAPE_DIR = os.path.join(CYTOSCAPE_ROOT_DIR, 'cytoscape')

def cd(directory=BUILD_PARENT_DIR, *subdirs):
    if subdirs:
        directory = os.path.join(directory, *subdirs)
    if os.getcwd() != directory:
        os.chdir(directory)

Changing to directory: /home/cybuilder/cytoscape-admin-scripts
Cloning into 'cytoscape'...
remote: Enumerating objects: 698, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 698 (delta 105), reused 141 (delta 79), pack-reused 529 (from 1)
Receiving objects: 100% (698/698), 793.96 KiB | 0 bytes/s, done.
Resolving deltas: 100% (358/358), done.


In [6]:
STARTING_BRANCH = 'release/3.10.4' 

## 2. Pull the develop branch of Cytoscape

Note that to execute the checkout step here, you will need to have set up an SSH key on this machine that does not use password validation, or else parts of the build will be stall when they require input. 

In [7]:
cd(CYTOSCAPE_ROOT_DIR)
![[ -d cytoscape ]] || ./cy.sh init
cd(CYTOSCAPE_DIR)
!./cy.sh run-all "git checkout {STARTING_BRANCH}"

Target directory = 
Cytoscape project will be cloned to: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape
Cloning into 'cytoscape'...
remote: Enumerating objects: 698, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 698 (delta 77), reused 96 (delta 54), pack-reused 577 (from 1)
Receiving objects: 100% (698/698), 797.93 KiB | 0 bytes/s, done.
Resolving deltas: 100% (361/361), done.
Cloning: parent (URI = git@github.com:cytoscape/cytoscape-parent.git)
Cloning into 'parent'...
remote: Enumerating objects: 1736, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 1736 (delta 63), reused 53 (delta 20), pack-reused 1640 (from 1)
Receiving objects: 100% (1736/1736), 267.79 KiB | 0 bytes/s, done.
Resolving deltas: 100% (691/691), done.
~/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/parent ~/cytoscape-admin-scripts/release-build/cytoscap

## 2a. Reset

In [8]:
cd(CYTOSCAPE_DIR)
!./cy.sh run-all 'git clean -f -d'
!./cy.sh run-all 'git reset --hard'

------------------------------------------------------------------------
Executing command: git clean -f -d
--in .
------------------------------------------------------------------------
--in parent
------------------------------------------------------------------------
--in api
------------------------------------------------------------------------
--in impl
------------------------------------------------------------------------
--in support
------------------------------------------------------------------------
--in gui-distribution
------------------------------------------------------------------------
--in app-developer
------------------------------------------------------------------------
------------------------------------------------------------------------
Executing command: git reset --hard
--in .
HEAD is now at dce4349 Set version to 3.10.4
------------------------------------------------------------------------
--in parent
HEAD is now at 3e3e3b3 Set version to 3.10.

## 2. Verify status

In [9]:
cd(CYTOSCAPE_DIR)
!./cy.sh run-all 'git status'

------------------------------------------------------------------------
Executing command: git status
--in .
# On branch release/3.10.4
nothing to commit, working directory clean
------------------------------------------------------------------------
--in parent
# On branch release/3.10.4
nothing to commit, working directory clean
------------------------------------------------------------------------
--in api
# On branch release/3.10.4
nothing to commit, working directory clean
------------------------------------------------------------------------
--in impl
# On branch release/3.10.4
nothing to commit, working directory clean
------------------------------------------------------------------------
--in support
# On branch release/3.10.4
nothing to commit, working directory clean
------------------------------------------------------------------------
--in gui-distribution
# On branch release/3.10.4
nothing to commit, working directory clean
---------------------------------------

## 3. Build Cytoscape and ensure no errors

This may take a while. Expect to build subrepos first before building from the root directory

Currently, you should expect two or so non-fatal javadoc-bundle-options related errors.

In [10]:
cd(CYTOSCAPE_DIR)
with open('build_output.txt', 'w') as outf:
    process = Popen('mvn clean install -Dmaven.test.skip=true'.split(' '), 
                stdout=outf,
                cwd=CYTOSCAPE_DIR)
    process.wait()

print("Showing ERROR lines in build...")
!cat build_output.txt | grep ERROR

Showing ERROR lines in build...
[ERROR] Error fetching link: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/api/app-api/target/javadoc-bundle-options. Ignored it.
[ERROR] Error fetching link: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/api/swing-app-api/target/javadoc-bundle-options. Ignored it.


## 4. Build Cytoscape installers

This requires Install4J to be configured on your machine and for the Install4j project file to point to the correct Mac signing file. The keystore password is set below (blank in this config), to be interpreted by the Install4j Maven plugin.

In [11]:
cd(CYTOSCAPE_DIR, 'gui-distribution', 'packaging')
%env MAC_KEYSTORE_PASSWORD=
!mvn clean install -U

env: MAC_KEYSTORE_PASSWORD=
[INFO] Scanning for projects...
[WARNING] 
[WARNING] Some problems were encountered while building the effective model for org.cytoscape.distribution:packaging:jar:3.10.4
[WARNING] 'build.plugins.plugin.version' for org.sonatype.install4j:install4j-maven-plugin is missing. @ org.cytoscape.distribution:packaging:[unknown-version], /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/pom.xml, line 138, column 15
[WARNING] 
[WARNING] It is highly recommended to fix these problems because they threaten the stability of your build.
[WARNING] 
[WARNING] For this reason, future Maven versions might no longer support building such malformed projects.
[WARNING] 
[INFO] 
[INFO] ----------------< org.cytoscape.distribution:packaging >----------------
[INFO] Building Cytoscape Release Packaging 3.10.4
[INFO] --------------------------------[ jar ]---------------------------------
Downloaded from central: https://repo.maven

[INFO]   Creating media file: 
[INFO]     Signing launcher
[INFO]     Zipping custom code & resources JAR file
[INFO]     Identifying components
[INFO]     Shrinking runtime
[INFO]   
[INFO] Creating media file 'unix':
[INFO]   Collecting files:
[INFO]   Compiling launchers:
[INFO]     Compiling launcher 'Cytoscape':
[INFO]       Generating launcher script file
[INFO]   Creating media file: 
[INFO]     Generating launcher script file
[INFO]     Zipping custom code & resources JAR file
[INFO]     Identifying components
[INFO]     Shrinking runtime
[INFO]   
[INFO] Compressing 4 media files with 4 threads:
[INFO] 
[INFO] Compressed media file 'unix':
[INFO]   Compressing files
[INFO]   Moving media files to media directory /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/media
[INFO]   The name of the media file is Cytoscape_3_10_4_unix.sh.
[INFO]   The size of the media file is 228.2 MB
[INFO] 
[INFO] Compressed media file 'wind

[INFO] 
[INFO] --- maven-install-plugin:2.4:install (default-install) @ packaging ---
[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/packaging-3.10.4.jar to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.10.4/packaging-3.10.4.jar
[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/pom.xml to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.10.4/packaging-3.10.4.pom
[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/packaging-3.10.4-sources.jar to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.10.4/packaging-3.10.4-sources.jar
[INFO] ------------------------------------------------------------------------
[INFO] BUILD SUCCESS
[INFO] ------------------------------------------------------------------------

## 5. Copying Cytoscape installers to weekly download page

When completed installer executables can be found in `cytoscape/cytoscape/gui-distribution/packaging/target/media` and compressed builds for Linux and Windows can be found in `cytoscape/cytoscape/gui-distribution/assembly/target`

all of which should be copied to `/var/www/html/cytoscape-builds/Cytoscape-3.9.0/<DATE>`



In [15]:
!mkdir -p /var/www/html/cytoscape-builds/Cytoscape-3.10.4/rc2
!rsync -av --progress /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/media/* /var/www/html/cytoscape-builds/Cytoscape-3.10.4/rc2
!rsync -av --progress /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/assembly/target/*.{gz,zip} /var/www/html/cytoscape-builds/Cytoscape-3.10.4/rc2


sending incremental file list

sent 256 bytes  received 12 bytes  536.00 bytes/sec
total size is 938,450,801  speedup is 3,501,682.09
sending incremental file list

sent 124 bytes  received 12 bytes  272.00 bytes/sec
total size is 721,039,897  speedup is 5,301,763.95


## 6. Notarizing on idekerlab-macmini

The install4j generated .dmg must be notarized to run on macOS 10.15 and above. The .dmg will be sent to idekerlab-macmini and submitted for notarization. An email will be sent to William Markuske when the notarization process is complete. As soon as the .dmg is notarized it will work and no further action is needed.

In [16]:
#!rsync -av --progress /var/www/html/cytoscape-builds/Cytoscape-3.9.0/$(date +%Y_%m_%d)/*.dmg idekerlab@idekerlab-macmini.ucsd.edu:~/apps_to_notarize/
#!ssh idekerlab@idekerlab-macmini.ucsd.edu '~/notarizedmg_jing.sh ~/apps_to_notarize/*.dmg Snapshot.$(date +%Y%m%d)'    

You can check the status of the notarization by pasting in the `RequestUUID` value into the following command and running.

In [14]:
#!ssh idekerlab@idekerlab-macmini.ucsd.edu '~/notarizestatus_jing.sh 6a41ff2a-9fd8-43d6-b378-adfd075b613b'